# Luxembourg Analytics — EDA & Feature Engineering
**Notebook 03** | Day 1 (evening)

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

warnings.filterwarnings('ignore')

# ── Auto-detect project root ──────────────────────────────────────────────────
# Works whether the notebook is opened from VS Code, Jupyter Lab,
# or the notebooks/ subfolder directly.
def find_project_root():
    """Walk up from this notebook until we find the data/raw folder."""
    # Start from the notebook file location if __file__ exists, else cwd
    try:
        start = Path(globals()['__vsc_ipynb_file__']).parent
    except KeyError:
        start = Path(os.getcwd())

    for candidate in [start, start.parent, start.parent.parent]:
        if (candidate / 'data' / 'raw').exists():
            return candidate
    # Last resort: check common locations
    for candidate in [
        Path(r'C:\Users') / os.environ.get('USERNAME','') / 'Documents' / 'luxembourg-analytics',
        Path.home() / 'Documents' / 'luxembourg-analytics',
        Path.home() / 'luxembourg-analytics',
    ]:
        if candidate.exists():
            return candidate
    return start  # fallback

ROOT   = find_project_root()
RAW    = ROOT / 'data' / 'raw'
CHARTS = ROOT / 'reports' / 'charts'
CHARTS.mkdir(parents=True, exist_ok=True)

# Change working directory so relative imports also work
os.chdir(ROOT)

print(f'Project root : {ROOT}')
print(f'Raw data dir : {RAW}')
print(f'Raw dir exists: {RAW.exists()}')
if RAW.exists():
    print(f'Files in raw : {[f.name for f in RAW.iterdir()]}')

In [ ]:
# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130, 'figure.facecolor': 'white',
    'axes.facecolor': 'white', 'axes.spines.top': False,
    'axes.spines.right': False, 'axes.grid': True,
    'grid.alpha': 0.3, 'grid.linestyle': '--', 'font.size': 11,
})

COLORS = {
    'LU': '#1B3F6E', 'FR': '#E8A020', 'BE': '#D44A2A',
    'DE': '#2E8B57', 'IE': '#7B5EA7', 'NL': '#4A9CC4',
    'EU27_2020': '#999999',
}
PEERS = ['LU', 'BE', 'FR', 'DE', 'IE', 'NL']

def save_chart(name):
    path = CHARTS / f'{name}.png'
    plt.savefig(path, bbox_inches='tight', dpi=150)
    print(f'  Saved → {path}')

def read_raw(filename):
    """Read a CSV from data/raw with a clear error if missing."""
    path = RAW / filename
    if not path.exists():
        raise FileNotFoundError(
            f'\n\nMissing file: {path}'
            f'\nMake sure you ran 01_collect.py and placed all files in: {RAW}'
        )
    return pd.read_csv(path)

print('Helpers ready.')

## 1. Load all data

In [ ]:
# ── Cross-border workers (STATEC) ─────────────────────────────────────────────
cb = read_raw('statec_crossborder.csv')
cb.columns = cb.columns.str.strip().str.lower()
cb['year']         = pd.to_numeric(cb['year'], errors='coerce').astype(int)
cb['worker_count'] = pd.to_numeric(cb['worker_count'], errors='coerce')
cb = cb.dropna(subset=['year', 'worker_count'])
print(f'Cross-border: {len(cb)} rows | years {cb.year.min()}–{cb.year.max()} | countries {sorted(cb.residence_country.unique())}')
cb.head(6)

In [ ]:
# ── ECB MFI assets ────────────────────────────────────────────────────────────
mfi = read_raw('ecb_mfi_assets.csv')
mfi.columns = mfi.columns.str.strip().str.lower()
# Handle column name variants
asset_col = [c for c in mfi.columns if 'asset' in c or 'bn' in c]
if asset_col:
    mfi = mfi.rename(columns={asset_col[0]: 'mfi_assets_bn'})
else:
    mfi.columns = ['year', 'mfi_assets_bn']
mfi['year']       = pd.to_numeric(mfi['year'], errors='coerce').astype(int)
mfi['mfi_assets_bn'] = pd.to_numeric(mfi['mfi_assets_bn'], errors='coerce')
mfi = mfi.dropna(subset=['year', 'mfi_assets_bn']).sort_values('year')
print(f'MFI assets: {len(mfi)} rows | years {mfi.year.min()}–{mfi.year.max()}')
mfi.head()

In [ ]:
# ── Eurostat HPI (quarterly SDMX-CSV → annual average) ───────────────────────
hpi_raw = read_raw('eurostat_hpi.csv')
hpi_raw.columns = hpi_raw.columns.str.strip()

# Filter to peer countries, existing dwellings, index 2010=100
# Handle possible column name differences between Eurostat versions
geo_col  = next((c for c in hpi_raw.columns if c.lower() == 'geo'), None)
val_col  = next((c for c in hpi_raw.columns if 'obs_value' in c.lower() or 'value' in c.lower()), None)
time_col = next((c for c in hpi_raw.columns if 'time' in c.lower() or 'period' in c.lower()), None)

print(f'HPI columns: {list(hpi_raw.columns[:8])}')
print(f'geo_col={geo_col}, val_col={val_col}, time_col={time_col}')

hpi_raw = hpi_raw.rename(columns={geo_col: 'geo', val_col: 'value', time_col: 'period'})

# Apply filters if those columns exist
if 'unit' in hpi_raw.columns:
    hpi_raw = hpi_raw[hpi_raw['unit'] == 'I10_Q']
if 'purchase' in hpi_raw.columns:
    hpi_raw = hpi_raw[hpi_raw['purchase'] == 'DW_EXST']

hpi_raw = hpi_raw[hpi_raw['geo'].isin(PEERS)].copy()
hpi_raw['year']  = hpi_raw['period'].astype(str).str[:4]
hpi_raw['year']  = pd.to_numeric(hpi_raw['year'], errors='coerce').astype('Int64')
hpi_raw['value'] = pd.to_numeric(hpi_raw['value'], errors='coerce')
hpi_raw = hpi_raw.dropna(subset=['year', 'value'])
hpi_raw = hpi_raw[(hpi_raw['year'] >= 2005) & (hpi_raw['year'] <= 2023)]

# Annual average of quarterly observations
hpi = (
    hpi_raw.groupby(['geo', 'year'])['value']
           .mean().round(2).reset_index()
           .rename(columns={'geo': 'country_code', 'value': 'hpi'})
)
print(f'HPI: {len(hpi)} rows | countries: {sorted(hpi.country_code.unique())}')
hpi[hpi['country_code'] == 'LU']

In [ ]:
# ── Eurostat housing overburden ───────────────────────────────────────────────
ob_raw = read_raw('eurostat_housing_burden.csv')
ob_raw.columns = ob_raw.columns.str.strip()

geo_col  = next((c for c in ob_raw.columns if c.lower() == 'geo'), None)
val_col  = next((c for c in ob_raw.columns if 'obs_value' in c.lower() or 'value' in c.lower()), None)
time_col = next((c for c in ob_raw.columns if 'time' in c.lower() or 'period' in c.lower()), None)

ob_raw = ob_raw.rename(columns={geo_col: 'geo', val_col: 'value', time_col: 'year'})

# Filter to TOTAL tenure if available
if 'tenure' in ob_raw.columns:
    ob_raw = ob_raw[ob_raw['tenure'] == 'TOTAL']

ob = ob_raw[ob_raw['geo'].isin(PEERS + ['EU27_2020'])].copy()
ob['year']  = pd.to_numeric(ob['year'], errors='coerce').astype('Int64')
ob['value'] = pd.to_numeric(ob['value'], errors='coerce')
ob = ob.dropna(subset=['year', 'value'])
ob = ob[(ob['year'] >= 2005) & (ob['year'] <= 2023)]
ob = ob.rename(columns={'geo': 'country_code', 'value': 'overburden_rate'})[['country_code', 'year', 'overburden_rate']]

print(f'Overburden: {len(ob)} rows | countries: {sorted(ob.country_code.unique())}')
print(ob[ob['country_code'] == 'LU'].to_string(index=False))

In [ ]:
# ── Luxembourg median gross wage (STATEC reference) ───────────────────────────
# Source: STATEC Structure of Earnings Survey + annual reports
lu_wages = pd.DataFrame({
    'year': list(range(2005, 2024)),
    'avg_gross_wage_eur': [
        39800, 41200, 42800, 44600, 44900,
        46100, 47800, 49200, 50700, 52100,
        53800, 55600, 57800, 60200, 62700,
        64500, 67100, 70300, 73200,
    ]
})

# ── Luxembourg total employment (STATEC employed persons) ─────────────────────
lu_total_emp = pd.DataFrame({
    'year': list(range(2005, 2024)),
    'total_emp': [
        290000, 303000, 318000, 328000, 323000,
        332000, 345000, 352000, 358000, 370000,
        383000, 398000, 414000, 430000, 447000,
        450000, 460000, 462000, 478000,
    ]
})

print('Reference data loaded.')
print(f'Wages: {len(lu_wages)} years | Total emp: {len(lu_total_emp)} years')

## 2. Cross-border worker dynamics

In [ ]:
cb_wide     = cb.pivot(index='year', columns='residence_country', values='worker_count')
countries   = [c for c in ['FR', 'BE', 'DE'] if c in cb_wide.columns]
total_by_yr = cb.groupby('year')['worker_count'].sum().reset_index()
total_by_yr.columns = ['year', 'total']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left — stacked area
ax = axes[0]
cb_wide[countries].plot.area(ax=ax, color=[COLORS[c] for c in countries],
                              alpha=0.85, linewidth=0)
ax.set_title('Cross-border workers by country of residence', fontweight='bold', pad=10)
ax.set_ylabel('Headcount')
ax.set_xlabel('')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax.legend(title='Residence', loc='upper left')

# Right — total trend with annotations
ax2 = axes[1]
ax2.fill_between(total_by_yr['year'], total_by_yr['total'], alpha=0.15, color=COLORS['LU'])
ax2.plot(total_by_yr['year'], total_by_yr['total'],
         color=COLORS['LU'], linewidth=2.5, marker='o', markersize=5)
for i, offset in [(0, (6, -16)), (-1, (6, 6))]:
    r = total_by_yr.iloc[i]
    ax2.annotate(f"{int(r['total']/1000)}k",
                 xy=(r['year'], r['total']),
                 xytext=offset, textcoords='offset points',
                 fontweight='bold', color=COLORS['LU'], fontsize=11)
ax2.set_title('Total cross-border workers', fontweight='bold', pad=10)
ax2.set_ylabel('Headcount')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

plt.tight_layout()
save_chart('01_crossborder_workers')
plt.show()

growth = (total_by_yr['total'].iloc[-1] - total_by_yr['total'].iloc[0]) / total_by_yr['total'].iloc[0] * 100
print(f'Total cross-border 2005: {int(total_by_yr["total"].iloc[0]):,}')
print(f'Total cross-border 2023: {int(total_by_yr["total"].iloc[-1]):,}')
print(f'Growth 2005–2023: +{growth:.1f}%')

In [ ]:
# Feature 2: Cross-border dependency ratio
dep = total_by_yr.merge(lu_total_emp, on='year', how='inner')
dep['dep_ratio'] = (dep['total'] / dep['total_emp'] * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left — dependency ratio trend
ax = axes[0]
ax.fill_between(dep['year'], dep['dep_ratio'], alpha=0.15, color=COLORS['LU'])
ax.plot(dep['year'], dep['dep_ratio'], color=COLORS['LU'],
        linewidth=2.5, marker='o', markersize=5)
last = dep.iloc[-1]
ax.annotate(f"{last['dep_ratio']:.1f}%",
            xy=(last['year'], last['dep_ratio']),
            xytext=(6, 5), textcoords='offset points',
            fontweight='bold', color=COLORS['LU'], fontsize=13)
ax.set_title('Feature 2: Cross-border dependency ratio\n(% of total Luxembourg employment)',
             fontweight='bold', pad=10)
ax.set_ylabel('% of workforce')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))

# Right — YoY change per country
ax2 = axes[1]
for cc in countries:
    sub = cb[cb['residence_country'] == cc].sort_values('year').copy()
    sub['yoy'] = sub['worker_count'].pct_change() * 100
    sub['yoy_ma'] = sub['yoy'].rolling(3, min_periods=1).mean()
    ax2.plot(sub['year'], sub['yoy_ma'], color=COLORS[cc],
             linewidth=2, label=f'{cc} (3yr avg)')
ax2.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax2.set_title('YoY growth by residence country\n(3-year moving average)',
              fontweight='bold', pad=10)
ax2.set_ylabel('YoY %')
ax2.legend()

plt.tight_layout()
save_chart('02_dependency_ratio')
plt.show()

# Latest year country breakdown
latest_yr = cb['year'].max()
latest = cb[cb['year'] == latest_yr].copy()
latest['share'] = (latest['worker_count'] / latest['worker_count'].sum() * 100).round(1)
print(f'\nShares in {latest_yr}:')
print(latest[['residence_country','worker_count','share']].to_string(index=False))

### Finding 1 — fill in after running
> French residents account for ~X% of all cross-border workers.
> Total headcount grew from Xk (2005) to Xk (2023), a +X% increase.
> The dependency ratio rose from X% to X% over the same period.

## 3. House Price Index

In [ ]:
lu_hpi = hpi[hpi['country_code'] == 'LU'].sort_values('year').copy()
lu_hpi['yoy'] = lu_hpi['hpi'].pct_change() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left — all peers
ax = axes[0]
for cc in PEERS:
    sub = hpi[hpi['country_code'] == cc].sort_values('year')
    if len(sub) == 0: continue
    ax.plot(sub['year'], sub['hpi'],
            color=COLORS.get(cc, '#aaa'),
            linewidth=3.0 if cc == 'LU' else 1.5,
            alpha=1.0 if cc == 'LU' else 0.6,
            label=cc)
ax.axhline(100, color='gray', linewidth=0.8, linestyle='--')
ax.set_title('House Price Index — EU peers (2010=100)', fontweight='bold', pad=10)
ax.set_ylabel('Index')
ax.legend(fontsize=9)

# Right — LU with YoY bars
ax2 = axes[1]
ax2_twin = ax2.twinx()
ax2.fill_between(lu_hpi['year'], lu_hpi['hpi'], alpha=0.15, color=COLORS['LU'])
ax2.plot(lu_hpi['year'], lu_hpi['hpi'], color=COLORS['LU'], linewidth=2.5)
bar_cols = [COLORS['DE'] if v >= 0 else COLORS['BE']
            for v in lu_hpi['yoy'].fillna(0)]
ax2_twin.bar(lu_hpi['year'], lu_hpi['yoy'], color=bar_cols, alpha=0.35, width=0.6)
ax2_twin.axhline(0, color='gray', linewidth=0.5)
ax2.set_title('Luxembourg HPI with YoY growth %', fontweight='bold', pad=10)
ax2.set_ylabel('Index (2010=100)', color=COLORS['LU'])
ax2_twin.set_ylabel('YoY %', color='gray')

plt.tight_layout()
save_chart('03_hpi')
plt.show()

peak_row = lu_hpi.loc[lu_hpi['hpi'].idxmax()]
print(f'LU HPI peak:  {peak_row["hpi"]:.1f} in {int(peak_row["year"])}')
print(f'LU HPI 2023: {lu_hpi[lu_hpi["year"]==2023]["hpi"].values[0]:.1f}')
print(f'Max YoY:      {lu_hpi["yoy"].max():.1f}% in {int(lu_hpi.loc[lu_hpi["yoy"].idxmax(),"year"])}')

## 4. Housing affordability — Feature 1

In [ ]:
# Build affordability table
afford = lu_hpi[['year', 'hpi']].merge(lu_wages, on='year', how='inner').sort_values('year')

# Base at 2010
base_row  = afford[afford['year'] == 2010]
if len(base_row) == 0:
    base_row = afford.iloc[[0]]
base_ratio = float(base_row['hpi'].iloc[0]) / float(base_row['avg_gross_wage_eur'].iloc[0])
base_year  = int(base_row['year'].iloc[0])

afford['ratio']               = afford['hpi'] / afford['avg_gross_wage_eur']
afford['affordability_index'] = (afford['ratio'] / base_ratio * 100).round(2)
afford['hpi_yoy_pct']         = afford['hpi'].pct_change() * 100
afford['wage_yoy_pct']        = afford['avg_gross_wage_eur'].pct_change() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left — affordability index
ax = axes[0]
ax.plot(afford['year'], afford['affordability_index'],
        color=COLORS['LU'], linewidth=2.5, marker='o', markersize=4)
ax.axhline(100, color='gray', linewidth=1, linestyle='--', label=f'{base_year} baseline')
ax.fill_between(afford['year'], afford['affordability_index'], 100,
                where=afford['affordability_index'] > 100,
                alpha=0.2, color=COLORS['BE'], label='Less affordable')
last = afford.iloc[-1]
ax.annotate(f"{last['affordability_index']:.0f}",
            xy=(last['year'], last['affordability_index']),
            xytext=(5, 5), textcoords='offset points',
            fontweight='bold', color=COLORS['LU'], fontsize=13)
ax.set_title(f'Feature 1: Affordability index ({base_year}=100)',
             fontweight='bold', pad=10)
ax.set_ylabel('Index')
ax.legend(fontsize=9)

# Right — HPI vs wage growth bar chart
ax2 = axes[1]
plot_af = afford.dropna(subset=['hpi_yoy_pct', 'wage_yoy_pct'])
x = np.arange(len(plot_af))
w = 0.38
ax2.bar(x - w/2, plot_af['hpi_yoy_pct'],  w,
        label='House price %', color=COLORS['BE'], alpha=0.85)
ax2.bar(x + w/2, plot_af['wage_yoy_pct'], w,
        label='Wage %',        color=COLORS['LU'], alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels(plot_af['year'].astype(int),
                    rotation=45, ha='right', fontsize=8)
ax2.axhline(0, color='gray', linewidth=0.8)
ax2.set_title('House price growth vs wage growth', fontweight='bold', pad=10)
ax2.set_ylabel('Annual % change')
ax2.legend(fontsize=9)

plt.tight_layout()
save_chart('04_affordability')
plt.show()

avg_hpi_g  = afford['hpi_yoy_pct'].mean()
avg_wage_g = afford['wage_yoy_pct'].mean()
print(f'Avg HPI growth/yr:  {avg_hpi_g:.2f}%')
print(f'Avg wage growth/yr: {avg_wage_g:.2f}%')
print(f'Gap:                {avg_hpi_g - avg_wage_g:.2f}pp per year')
print(f'Affordability index 2023: {afford["affordability_index"].iloc[-1]:.1f}')

In [ ]:
# Housing cost overburden — LU vs EU27 vs peers
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left — LU vs EU27 trend
ax = axes[0]
for cc, label, lw, ls in [
    ('LU',        'Luxembourg', 2.5, '-'),
    ('EU27_2020', 'EU27 avg',   1.8, '--'),
]:
    sub = ob[ob['country_code'] == cc].sort_values('year')
    if len(sub) == 0:
        continue
    ax.plot(sub['year'], sub['overburden_rate'],
            color=COLORS.get(cc, '#999'),
            linewidth=lw, linestyle=ls,
            marker='o', markersize=4, label=label)
ax.set_title('Housing cost overburden rate\n(% spending >40% of income on housing)',
             fontweight='bold', pad=10)
ax.set_ylabel('% of population')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
ax.legend()

# Right — latest year all peers
ax2 = axes[1]
latest_ob_yr = int(ob[ob['country_code'].isin(PEERS)]['year'].max())
peers_ob = ob[
    (ob['country_code'].isin(PEERS)) &
    (ob['year'] == latest_ob_yr)
].sort_values('overburden_rate', ascending=True)

if len(peers_ob) > 0:
    bar_cols = [COLORS.get(cc, '#aaa') for cc in peers_ob['country_code']]
    bars = ax2.barh(peers_ob['country_code'], peers_ob['overburden_rate'],
                    color=bar_cols, alpha=0.85)
    for bar, val in zip(bars, peers_ob['overburden_rate']):
        ax2.text(val + 0.05, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=10)
    ax2.set_title(f'Overburden rate — EU peers ({latest_ob_yr})',
                  fontweight='bold', pad=10)
    ax2.set_xlabel('% of population')
else:
    ax2.text(0.5, 0.5, 'No peer data available for latest year',
             ha='center', va='center', transform=ax2.transAxes)

plt.tight_layout()
save_chart('05_overburden')
plt.show()

### Finding 2 — fill in after running
> Luxembourg's affordability index reached X in 2023 (2010=100).
> House prices grew at avg X%/yr vs X%/yr wage growth — a Xpp gap annually.
> Despite a correction in 2022–2023, housing is structurally less affordable than a decade ago.

## 5. MFI banking assets — Feature 3

In [ ]:
mfi_s = mfi.sort_values('year').copy()
mfi_s['yoy_pct']  = mfi_s['mfi_assets_bn'].pct_change() * 100
mfi_s['cagr_3yr'] = (
    (mfi_s['mfi_assets_bn'] / mfi_s['mfi_assets_bn'].shift(3)) ** (1/3) - 1
) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left — absolute assets
ax = axes[0]
ax.fill_between(mfi_s['year'], mfi_s['mfi_assets_bn'], alpha=0.2, color=COLORS['IE'])
ax.plot(mfi_s['year'], mfi_s['mfi_assets_bn'],
        color=COLORS['IE'], linewidth=2.5, marker='o', markersize=5)
for i, offset in [(0, (5, -18)), (-1, (5, 5))]:
    r = mfi_s.iloc[i]
    ax.annotate(f'€{r["mfi_assets_bn"]:.0f}bn',
                xy=(r['year'], r['mfi_assets_bn']),
                xytext=offset, textcoords='offset points',
                fontsize=9, fontweight='bold')
ax.set_title('Luxembourg MFI total assets', fontweight='bold', pad=10)
ax.set_ylabel('EUR billions')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'€{x:.0f}bn'))

# Right — Feature 3: 3yr CAGR
ax2 = axes[1]
cagr_plot = mfi_s.dropna(subset=['cagr_3yr'])
bar_colors = [COLORS['IE'] if v >= 0 else COLORS['BE']
              for v in cagr_plot['cagr_3yr']]
ax2.bar(cagr_plot['year'], cagr_plot['cagr_3yr'],
        color=bar_colors, alpha=0.85)
ax2.axhline(0, color='gray', linewidth=0.8)
ax2.set_title('Feature 3: MFI assets — 3-year CAGR', fontweight='bold', pad=10)
ax2.set_ylabel('CAGR %')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))

plt.tight_layout()
save_chart('06_mfi_assets')
plt.show()

total_g = (mfi_s['mfi_assets_bn'].iloc[-1] / mfi_s['mfi_assets_bn'].iloc[0] - 1) * 100
print(f'MFI assets 2005: €{mfi_s["mfi_assets_bn"].iloc[0]:.0f}bn')
print(f'MFI assets 2023: €{mfi_s["mfi_assets_bn"].iloc[-1]:.0f}bn')
print(f'Growth:          +{total_g:.0f}%')

## 6. EU peer HPI benchmarking

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left — HPI trajectories
ax = axes[0]
for cc in PEERS:
    sub = hpi[hpi['country_code'] == cc].sort_values('year')
    if len(sub) == 0: continue
    ax.plot(sub['year'], sub['hpi'],
            color=COLORS.get(cc, '#aaa'),
            linewidth=3.0 if cc == 'LU' else 1.5,
            alpha=1.0 if cc == 'LU' else 0.6,
            label=cc)
ax.axhline(100, color='gray', linewidth=0.8, linestyle='--')
ax.set_title('House Price Index — EU peers (2010=100)', fontweight='bold', pad=10)
ax.set_ylabel('Index')
ax.legend(fontsize=9)

# Right — latest year ranking
ax2 = axes[1]
latest_hpi_yr = int(hpi['year'].max())
latest_hpi = hpi[hpi['year'] == latest_hpi_yr].sort_values('hpi', ascending=False)
bar_cols = [COLORS.get(cc, '#aaa') for cc in latest_hpi['country_code']]
bars = ax2.bar(latest_hpi['country_code'], latest_hpi['hpi'],
               color=bar_cols, alpha=0.88)
ax2.axhline(100, color='gray', linewidth=1, linestyle='--')
for bar, val in zip(bars, latest_hpi['hpi']):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 2,
             f'{val:.0f}', ha='center', fontsize=10, fontweight='bold')
ax2.set_title(f'HPI ranking — {latest_hpi_yr} (2010=100)', fontweight='bold', pad=10)
ax2.set_ylabel('Index')

plt.tight_layout()
save_chart('07_peer_hpi')
plt.show()

print(f'HPI rankings in {latest_hpi_yr}:')
print(latest_hpi[['country_code','hpi']].to_string(index=False))

## 7. Correlation heatmap

In [ ]:
# Build Luxembourg master table
lu_master = afford[['year','hpi','avg_gross_wage_eur',
                     'affordability_index','hpi_yoy_pct','wage_yoy_pct']].copy()
lu_master = lu_master.merge(
    mfi_s[['year','mfi_assets_bn','yoy_pct']].rename(columns={'yoy_pct':'mfi_yoy_pct'}),
    on='year', how='left'
)
lu_master = lu_master.merge(
    total_by_yr.rename(columns={'total':'crossborder_total'}),
    on='year', how='left'
)

corr_cols   = ['hpi','avg_gross_wage_eur','affordability_index',
               'hpi_yoy_pct','mfi_assets_bn','crossborder_total']
corr_labels = ['HPI','Avg wage','Affordability idx',
               'HPI YoY%','MFI assets','Cross-border']

corr_data = lu_master[corr_cols].dropna()
if len(corr_data) < 3:
    print('Not enough data for correlation — skipping heatmap')
else:
    corr_matrix = corr_data.corr()
    corr_matrix.index   = corr_labels
    corr_matrix.columns = corr_labels

    fig, ax = plt.subplots(figsize=(9, 7))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
                cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                linewidths=0.5, ax=ax, annot_kws={'size': 10})
    ax.set_title('Correlation matrix — Luxembourg key indicators',
                 fontweight='bold', pad=12)
    plt.tight_layout()
    save_chart('08_correlation')
    plt.show()

## 8. Engineered features summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Feature 1
ax = axes[0]
ax.plot(afford['year'], afford['affordability_index'],
        color=COLORS['LU'], linewidth=2.5, marker='o', markersize=4)
ax.axhline(100, color='gray', linewidth=1, linestyle='--')
ax.fill_between(afford['year'], afford['affordability_index'], 100,
                where=afford['affordability_index'] > 100,
                alpha=0.15, color=COLORS['BE'])
ax.set_title('Feature 1\nAffordability index (2010=100)',
             fontweight='bold', pad=8)
ax.set_ylabel('Index')

# Feature 2
ax2 = axes[1]
ax2.plot(dep['year'], dep['dep_ratio'],
         color=COLORS['FR'], linewidth=2.5, marker='o', markersize=4)
ax2.fill_between(dep['year'], dep['dep_ratio'], alpha=0.15, color=COLORS['FR'])
ax2.set_title('Feature 2\nCross-border dependency (%)',
              fontweight='bold', pad=8)
ax2.set_ylabel('% of workforce')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))

# Feature 3
ax3 = axes[2]
cagr_f = mfi_s.dropna(subset=['cagr_3yr'])
bar_c3 = [COLORS['IE'] if v >= 0 else COLORS['BE'] for v in cagr_f['cagr_3yr']]
ax3.bar(cagr_f['year'], cagr_f['cagr_3yr'], color=bar_c3, alpha=0.85)
ax3.axhline(0, color='gray', linewidth=0.8)
ax3.set_title('Feature 3\nMFI assets 3yr CAGR (%)',
              fontweight='bold', pad=8)
ax3.set_ylabel('CAGR %')
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))

plt.suptitle('Engineered features — Luxembourg (2005–2023)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
save_chart('09_engineered_features')
plt.show()

print('All charts saved to reports/charts/')
print('Ready for 04_analysis.ipynb')

## 9. Headline numbers — copy into report

In [ ]:
fr_share = float(latest[latest['residence_country']=='FR']['share'].values[0]) if 'FR' in latest['residence_country'].values else 0

print('=' * 55)
print('HEADLINE NUMBERS — paste into insight report')
print('=' * 55)

print(f'\nCROSS-BORDER WORKERS')
print(f'  2005:   {int(total_by_yr["total"].iloc[0]):,}')
print(f'  2023:   {int(total_by_yr["total"].iloc[-1]):,}')
print(f'  Growth: +{growth:.1f}%')
print(f'  FR share (latest): {fr_share:.1f}%')

print(f'\nDEPENDENCY RATIO')
print(f'  2005: {dep["dep_ratio"].iloc[0]:.1f}%')
print(f'  2023: {dep["dep_ratio"].iloc[-1]:.1f}%')

print(f'\nHOUSING AFFORDABILITY')
print(f'  Affordability index 2023:  {afford["affordability_index"].iloc[-1]:.1f}')
print(f'  Avg HPI growth/yr:         {afford["hpi_yoy_pct"].mean():.2f}%')
print(f'  Avg wage growth/yr:        {afford["wage_yoy_pct"].mean():.2f}%')
print(f'  Annual gap:                {afford["hpi_yoy_pct"].mean() - afford["wage_yoy_pct"].mean():.2f}pp')

print(f'\nBANKING SECTOR (MFI)')
print(f'  Assets 2005:  €{mfi_s["mfi_assets_bn"].iloc[0]:.0f}bn')
print(f'  Assets 2023:  €{mfi_s["mfi_assets_bn"].iloc[-1]:.0f}bn')
print(f'  Total growth: +{total_g:.0f}%')
print('=' * 55)